In [1]:
print("hi")

hi


In [2]:
import json
import os
import re
import cv2
import numpy as np
import pdfplumber
import torch
from pathlib import Path
from PIL import Image
# from pipe_fn import pipe
from transformers import pipeline
from output_utils import save_split_output
from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)
# =========================================================
# LOAD MODEL
# =========================================================

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)

# ═══════════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ═══════════════════════════════════════════════════════════════════════════════

MAX_RETRIES = 2

SERVICE_CODE_RE = re.compile(r'^[A-Z]\d{4}$')

AMOUNT_FIELDS = [
    "total_charge",
    "not_covered",
    "in_network_adjustment",
    "deductible",
    "co_pay",
    "coinsurance",
]

DISPLAY_NAMES = {
    "total_charge":          "Total Charge",
    "not_covered":           "Not Covered",
    "in_network_adjustment": "In-Network Adjustment",
    "deductible":            "Deductible",
    "co_pay":                "Co-Pay",
    "coinsurance":           "Coinsurance",
}


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — PDF TABLE CROPPING
# ═══════════════════════════════════════════════════════════════════════════════

def normalize_text(text: str) -> str:
    """Collapse repeated characters: PPPPaaaattttiiiieeeennnntttt → Patient."""
    return re.sub(r"(.)\1+", r"\1", text)


def extract_starts_and_ends(page):
    """
    Scan one PDF page for:
      starts → y-top  of every 'Patient Detailed Claim Breakdown' header
      ends   → y-bottom of every 'Patient Responsibility' label
    """
    words  = page.extract_words()
    starts = []
    ends   = []

    for i in range(len(words) - 2):
        w1 = normalize_text(words[i]["text"])
        w2 = normalize_text(words[i + 1]["text"])
        w3 = normalize_text(words[i + 2]["text"])
        if "Patient" in w1 and "Detailed" in w2 and "Claim" in w3:
            starts.append(words[i]["top"])

    for i in range(len(words) - 1):
        w1 = normalize_text(words[i]["text"])
        w2 = normalize_text(words[i + 1]["text"])
        if "Patient" in w1 and "Responsibility" in w2:
            ends.append(words[i + 1]["bottom"])

    return starts, ends


def count_service_rows(page, region_top: float, region_bottom: float) -> int:
    """Count distinct service rows by ADA procedure codes (e.g. D0120) inside a y-region."""
    service_rows = set()
    for w in page.extract_words():
        if SERVICE_CODE_RE.fullmatch(w["text"].strip()):
            y = float(w["top"])
            if region_top <= y <= region_bottom:
                service_rows.add(round(y, 1))
    return len(service_rows)


def crop_page_region(page, top_y: float, bottom_y: float,
                     padding_top: int = 10, padding_bottom: int = 20) -> Image.Image:
    """Crop a pdfplumber page to [top_y, bottom_y] and return a PIL Image."""
    bbox = (
        0,
        max(0, top_y - padding_top),
        page.width,
        min(page.height, bottom_y + padding_bottom),
    )
    return page.crop(bbox).to_image(resolution=300).original


def stitch_images_vertically(images: list) -> Image.Image:
    """Stack PIL Images top-to-bottom into a single image."""
    total_width  = max(img.width  for img in images)
    total_height = sum(img.height for img in images)
    stitched = Image.new("RGB", (total_width, total_height), color=(255, 255, 255))
    y_offset = 0
    for img in images:
        stitched.paste(img, (0, y_offset))
        y_offset += img.height
    return stitched


def crop_medcost_claim_tables(pdf_path: str, output_dir: str = "cropped_tables_medcost") -> list:
    """
    Crop every Medcost claim table from the PDF.

    Returns a list of dicts:
        [{"image_path": str, "expected_rows": int, "claim_idx": int}, ...]

    Cross-page tables are automatically detected and stitched with PIL.
    """
    os.makedirs(output_dir, exist_ok=True)
    results = []

    with pdfplumber.open(pdf_path) as pdf:
        num_pages = len(pdf.pages)

        # ── Step 1: scan every page for header / footer markers ───────────────
        page_data = {}
        for page_num, page in enumerate(pdf.pages, start=1):
            starts, ends = extract_starts_and_ends(page)
            page_data[page_num] = {"page": page, "starts": starts, "ends": ends}
            print(f"  Page {page_num} → starts: {starts}  ends: {ends}")

        # ── Step 2: build flat ordered lists across all pages ─────────────────
        all_starts = []
        all_ends   = []
        for pg in range(1, num_pages + 1):
            for y in page_data[pg]["starts"]:
                all_starts.append((pg, y))
            for y in page_data[pg]["ends"]:
                all_ends.append((pg, y))

        # ── Step 3: greedily pair each start with the next available end ──────
        used_ends = set()

        for claim_idx, (start_pg, start_y) in enumerate(all_starts, start=1):
            matched_end = None
            for end_key, (end_pg, end_y) in enumerate(all_ends):
                if end_key in used_ends:
                    continue
                if end_pg > start_pg or (end_pg == start_pg and end_y > start_y):
                    matched_end = (end_key, end_pg, end_y)
                    break

            if matched_end is None:
                print(f"  ⚠  No end found for claim {claim_idx} (page {start_pg})")
                continue

            end_key, end_pg, end_y = matched_end
            used_ends.add(end_key)
            print(f"  Claim {claim_idx}: p{start_pg} y={start_y:.0f} → p{end_pg} y={end_y:.0f}")

            # ── Step 4: collect page-slice images ─────────────────────────────
            images = []

            if start_pg == end_pg:
                images.append(
                    crop_page_region(page_data[start_pg]["page"], start_y, end_y)
                )
            else:
                # First page: start_y → bottom
                fp = page_data[start_pg]["page"]
                images.append(crop_page_region(fp, start_y, fp.height, padding_bottom=0))

                # Middle pages (rare but possible): full page
                for mid_pg in range(start_pg + 1, end_pg):
                    mp = page_data[mid_pg]["page"]
                    images.append(crop_page_region(mp, 0, mp.height, padding_top=0, padding_bottom=0))

                # Last page: top → end_y
                lp = page_data[end_pg]["page"]
                images.append(crop_page_region(lp, 0, end_y, padding_top=0))

            # ── Step 5: stitch if multi-page ──────────────────────────────────
            if len(images) > 1:
                print(f"    ↳ Stitching {len(images)} slices")
                final_img = stitch_images_vertically(images)
            else:
                final_img = images[0]

            # ── Step 6: count service rows for later validation ────────────────
            # We count across all involved pages in the matched region
            expected_rows = 0
            if start_pg == end_pg:
                expected_rows = count_service_rows(
                    page_data[start_pg]["page"], start_y, end_y
                )
            else:
                expected_rows += count_service_rows(
                    page_data[start_pg]["page"], start_y, page_data[start_pg]["page"].height
                )
                for mid_pg in range(start_pg + 1, end_pg):
                    mp = page_data[mid_pg]["page"]
                    expected_rows += count_service_rows(mp, 0, mp.height)
                expected_rows += count_service_rows(page_data[end_pg]["page"], 0, end_y)

            # ── Step 7: save ───────────────────────────────────────────────────
            output_path = os.path.join(output_dir, f"page{start_pg}_claim{claim_idx}.png")
            final_img.save(output_path)
            print(f"    ✔  Saved: {output_path}  (expected_rows={expected_rows})")

            results.append({
                "image_path":    output_path,
                "expected_rows": expected_rows,
                "claim_idx":     claim_idx,
            })

    return results


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — IMAGE ENHANCEMENT
# ═══════════════════════════════════════════════════════════════════════════════

def enhance_table_image(image_path: str) -> Image.Image:
    """
    Strengthen faint horizontal lines in the table so the VLM reads rows cleanly.
    Returns a PIL Image (RGB).
    """
    img  = cv2.imread(image_path)
    gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)
    sums      = np.sum(thresh, axis=1)
    threshold = (thresh.shape[1] * 255) * 0.6
    lines     = np.where(sums > threshold)[0]

    for line_y in lines:
        cv2.line(img, (0, line_y), (thresh.shape[1], line_y), (0, 0, 0), 1)

    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — PROMPT
# ═══════════════════════════════════════════════════════════════════════════════
def build_prompt(eob_id: str) -> str:
    return f"""
You are extracting data from a Medcost Dental Explanation of Benefits (EOB) table image.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CRITICAL SCANNING INSTRUCTIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. Scan the ENTIRE image from top to bottom before generating output.

2. The claim table may be split into multiple sections on the same page.

3. The table may continue:
   - below a large blank area
   - below page numbers
   - below barcodes or QR codes
   - below footers
   - below repeated table headers

4. Repeated occurrences of:
   "Patient Detailed Claim Breakdown"
   DO NOT indicate a new patient.

5. If the same patient name and claim number appear again lower on the page,
   treat it as a continuation of the SAME claim.

6. Extract service rows from ALL table sections and merge them into a single
   services array.

7. Preserve the exact top-to-bottom order of rows as they appear in the image.

8. Do NOT stop after the first table section.

9. Continue scanning until:
   - the Totals row is found
   OR
   - the Summary block is found.

10. Before generating JSON:
    - Identify every visible service row.
    - Count all service rows.
    - Verify every row is included.
    - Then generate JSON.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EXTRACTION RULES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. Header fields to extract:
   - patient_name -> Patient Name:
   - provider -> Provider

2. Extract EVERY visible service row.
   Do NOT skip rows.
   Do NOT skip duplicate service codes.
   Do NOT deduplicate rows.

3. For each service row extract:

   - service_date
   - service_code
   - total_charge
   - in_network_adjustment
   - not_covered
   - deductible
   - co_pay
   - coinsurance
   - plan_payment_amount

4. Extract the Totals row into a separate "totals" object.

5. Also extract this in totals block:

   - provider_paid_amount -> plan_payment_amount
   - employee_paid_amount
   - total_payment
   - patient_responsibility

6. Amount handling:

   - Remove "$"
   - Remove commas
   - Return plain numeric strings
   - Example: "197.41"

7. Blank monetary cells:

   - If blank or shown as zero, return "0.00"

8. Never calculate values.
   Extract only what is visible.

9. If a value is not visible, return an empty string.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FIELD-LEVEL CONFIDENCE RULES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

EVERY extracted field MUST use this exact structure:

{{
    "value": "",
    "confidence": 0.0
}}

NEVER return an extracted field as a plain string.

WRONG:

"service_code": "D0120"

CORRECT:

"service_code": {{
    "value": "D0120",
    "confidence": 0.99
}}

Confidence rules:

- confidence MUST be a number between 0.0 and 1.0.
- 1.0 = completely certain that the value was correctly read.
- 0.0 = value is missing, unreadable, or cannot be reliably extracted.
- Do NOT guess values.
- If a value cannot be reliably extracted, return:

{{
    "value": "",
    "confidence": 0.0
}}

- Confidence represents ONLY confidence in reading the value from
  the image.
- Do NOT calculate confidence based on the financial amount.
- Do NOT calculate confidence using validation results.
- Do NOT use row-count validation to determine field confidence.

EVERY field in:
- patient_name
- provider
- every service row
- every totals field

MUST contain both:
- value
- confidence

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT FORMAT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Return ONLY valid JSON.

No explanation.
No markdown.
No extra text.
No comments.

The JSON MUST follow this structure:

{{
    "patient_name": {{
        "value": "",
        "confidence": 0.0
    }},

    "provider": {{
        "value": "",
        "confidence": 0.0
    }},

    "services": [
        {{
            "service_date": {{
                "value": "",
                "confidence": 0.0
            }},

            "service_code": {{
                "value": "",
                "confidence": 0.0
            }},

            "total_charge": {{
                "value": "",
                "confidence": 0.0
            }},

            "in_network_adjustment": {{
                "value": "",
                "confidence": 0.0
            }},

            "not_covered": {{
                "value": "",
                "confidence": 0.0
            }},

            "deductible": {{
                "value": "",
                "confidence": 0.0
            }},

            "co_pay": {{
                "value": "",
                "confidence": 0.0
            }},

            "coinsurance": {{
                "value": "",
                "confidence": 0.0
            }},

            "plan_payment_amount": {{
                "value": "",
                "confidence": 0.0
            }}
        }}
    ],

    "totals": {{
        "total_charge": {{
            "value": "",
            "confidence": 0.0
        }},

        "in_network_adjustment": {{
            "value": "",
            "confidence": 0.0
        }},

        "not_covered": {{
            "value": "",
            "confidence": 0.0
        }},

        "deductible": {{
            "value": "",
            "confidence": 0.0
        }},

        "co_pay": {{
            "value": "",
            "confidence": 0.0
        }},

        "coinsurance": {{
            "value": "",
            "confidence": 0.0
        }},

        "plan_payment_amount": {{
            "value": "",
            "confidence": 0.0
        }},

        "employee_paid_amount": {{
            "value": "",
            "confidence": 0.0
        }},

        "total_payment": {{
            "value": "",
            "confidence": 0.0
        }},

        "patient_responsibility": {{
            "value": "",
            "confidence": 0.0
        }}
    }}
}}

 For every extracted field, return:
   - value
   - confidence
 
VALUE + CONFIDENCE RULES:
 
For every field return:
{{
  "value": "",
  "confidence": ""
}}
 
VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""
 
CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.
 
IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.
 
If value = "":
confidence MUST = 0.0.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FINAL CHECK BEFORE RESPONDING
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Before returning the JSON, verify:

1. EVERY field contains:
   - "value"
   - "confidence"

2. NO field is returned as a plain string.

3. Every service row follows the exact structure above.

4. Every totals field follows the exact structure above.

5. Duplicate service rows are preserved.

6. The Totals row is NOT included inside services.

7. All visible service rows are included.

8. Do NOT add fields that are not defined in the schema.

9. Return ONLY valid JSON.

"""
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — AMOUNT HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

def parse_amount(value) -> float:
    """Safely parse a monetary string to float."""
    if value is None or str(value).strip() in ("", "N/A"):
        return 0.0
    try:
        return float(str(value).replace("$", "").replace(",", "").strip())
    except ValueError:
        return 0.0


def format_amount(value) -> str:
    """Return a clean 2-decimal string for any monetary value."""
    try:
        return f"{float(str(value).replace('$','').replace(',','').strip()):.2f}"
    except Exception:
        return "0.00"


def normalize_amounts(obj):
    """
    Recursively walk the extracted dict and format every known amount field.
    """
    if isinstance(obj, dict):
        return {
            k: format_amount(v) if k in AMOUNT_FIELDS else normalize_amounts(v)
            for k, v in obj.items()
        }
    if isinstance(obj, list):
        return [normalize_amounts(i) for i in obj]
    return obj


def compute_totals_from_services(services: list) -> dict:
    """Sum each amount field across all service rows."""
    return {
        f: round(sum(parse_amount(s.get(f, "")) for s in services), 2)
        for f in AMOUNT_FIELDS
    }


def services_are_empty(services: list) -> bool:
    """Return True when every amount in every service row is zero/blank."""
    for svc in services:
        for f in AMOUNT_FIELDS:
            if str(svc.get(f, "")).strip() not in ("", "0.00", "0"):
                return False
    return True


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — JSON CLEANING
# ═══════════════════════════════════════════════════════════════════════════════

def extract_json(raw_text: str) -> dict:
    """
    Pull the first complete JSON object out of raw model output.
    Strips markdown fences, then finds the outermost { … }.
    """
    # Strip ```json ... ``` or ``` ... ```
    cleaned = re.sub(r"```(?:json)?", "", raw_text).strip()
    start = cleaned.find("{")
    end   = cleaned.rfind("}") + 1
    if start == -1 or end == 0:
        raise ValueError("No JSON object found in model output")
    return json.loads(cleaned[start:end])


def normalise_keys(parsed: dict) -> dict:
    """Fix common key typos produced by the model."""
    rename_map = {
        "Realationship":      "relationship",
        "realationship":      "relationship",
        "patientName":        "patient_name",
        "Provider":           "provider",
        "datesOfService":     "dates_of_service",
        "planPaymentAmount":  "plan_payment_amount",
        "totalCharge":        "total_charge",
        "notCovered":         "not_covered",
        "inNetworkAdjustment":"in_network_adjustment",
        "coPay":              "co_pay",
    }
    corrected = {}
    for k, v in parsed.items():
        corrected[rename_map.get(k, k)] = v
    return corrected


def save_json(data, output_path: str):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"  💾 JSON saved → {output_path}")


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════

def validate_claim(patient: dict, expected_row_count: int) -> tuple:
    """
    Validate one extracted claim dict.

    Returns:
        (is_valid: bool, errors: list[dict])
    """
    patient_name = patient.get("patient_name", "UNKNOWN")
    provider = patient.get("provider", "UNKNOWN")
    services     = patient.get("services", [])
    totals       = patient.get("totals",   {})
    errors       = []
    has_error    = False

    total_fields = len(AMOUNT_FIELDS)

    # ── Guard: no services at all ─────────────────────────────────────────────
    if not services:
        print(f"  ❌ [{patient_name}] No services found")
        field_errors = [{"field": f, "computed": 0.0, "extracted": None} for f in AMOUNT_FIELDS]
        return False, [{"type": "empty_services"}] + field_errors, total_fields

    # ── Guard: all amounts blank ───────────────────────────────────────────────
    if services_are_empty(services):
        msg = "All service amount fields are empty — model extraction likely failed"
        print(f"  ❌ [{patient_name}] {msg}")
        field_errors = [{"field": f, "computed": 0.0, "extracted": None} for f in AMOUNT_FIELDS]
        return False, [{"type": "all_service_rows_empty", "message": msg}] + field_errors, total_fields

    # ── Row count check ────────────────────────────────────────────────────────
    extracted_count = len(services)
    row_match       = (expected_row_count == extracted_count)

    print(f"\n  📊 Row Count [{patient_name}]")
    print(f"  {'✅' if row_match else '❌'}  expected={expected_row_count}  extracted={extracted_count}")

    if not row_match:
        has_error = True
        errors.append({
            "type":           "row_count_mismatch",
            "expected_rows":  expected_row_count,
            "extracted_rows": extracted_count,
        })

    # ── Totals cross-check ────────────────────────────────────────────────────
    computed = compute_totals_from_services(services)

    print(f"\n  🔍 Totals Validation [{patient_name}]")
    print(f"  {'Field':<28} {'Computed':>10}  {'Extracted':>10}  Status")
    print("  " + "─" * 60)

    for field, computed_val in computed.items():
        extracted_val = round(parse_amount(totals.get(field, "")), 2)
        diff          = round(computed_val - extracted_val, 2)
        match         = abs(diff) <= 0.01
        label         = DISPLAY_NAMES.get(field, field)
        icon          = "✅" if match else "❌"

        print(f"  {icon} {label:<26} {computed_val:>10.2f}  {extracted_val:>10.2f}  {'OK' if match else f'Δ {diff:+.2f}'}")

        if not match:
            has_error = True
            errors.append({
                "type":      "totals_mismatch",
                "field":     field,
                "computed":  computed_val,
                "extracted": extracted_val,
                "diff":      diff,
            })

    print("  " + "─" * 60)
    status = "PASSED ✅" if not has_error else "FAILED ❌"
    print(f"  [{patient_name}] Validation {status}\n")

    return not has_error, errors, total_fields
def clean_date_of_service(obj):

    if isinstance(obj, dict):
        new_obj = {}

        for k, v in obj.items():

            if k in ("service_date", "date_of_service") and isinstance(v, str):
                new_obj[k] = re.split(r"\s*-\s*", v.strip())[0]
            else:
                new_obj[k] = clean_date_of_service(v)

        return new_obj

    elif isinstance(obj, list):
        return [clean_date_of_service(i) for i in obj]

    return obj

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 7 — DENIAL CHECK
# ═══════════════════════════════════════════════════════════════════════════════

def check_claim_denied(pdf_path: str) -> str:
    """Scan PDF text for denial keywords (stops at 'Comments' section)."""
    denial_keywords = ["denied", "denial"]
    stop_phrase     = "Reason Code Description"

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if not text:
                continue
            searchable = text.lower()
            if stop_phrase in searchable:
                searchable = searchable.split(stop_phrase)[0]
            for kw in denial_keywords:
                if kw in searchable:
                    print(f"  ⚠  Denial keyword found: '{kw}'")
                    return "denied"

    return "not denied"


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 8 — VLM INFERENCE WITH RETRY
# ═══════════════════════════════════════════════════════════════════════════════

def run_model_with_retry(image: Image.Image, prompt: str, expected_rows: int) -> dict | None:
    """
    Call the VLM up to MAX_RETRIES times.

    Accepts a result only when:
      - JSON parses cleanly
      - Row count matches expected_rows
      - Not all service amounts are empty

    Falls back to the last parseable result if retries are exhausted.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text":  prompt},
            ],
        }
    ]

    last_parsed = None

    for attempt in range(1, MAX_RETRIES + 1):
        print(f"  🔄 Model attempt {attempt}/{MAX_RETRIES} …")

        with torch.no_grad():
            output = pipe(
                messages,
                max_new_tokens=15000,
                temperature=0.0,
                do_sample=False,
            )

        raw = output[0]["generated_text"]
        if isinstance(raw, list):
            raw = raw[-1]["content"]

        # ── Parse ─────────────────────────────────────────────────────────────
        try:
            parsed = extract_json(raw)
        except Exception as exc:
            print(f"    ❌ JSON parse failed: {exc}")
            continue

        model_confidence = calculate_model_confidence(parsed)
        parsed = _unwrap_vlm_output(parsed)
        parsed["_model_confidence"] = model_confidence

        parsed = normalise_keys(parsed)
        parsed = clean_date_of_service(parsed)

        services  = parsed.get("services", [])
        row_ok    = (len(services) == expected_rows)
        empty_ok  = not services_are_empty(services)

        if row_ok and empty_ok:
            print(f"    ✅ Accepted on attempt {attempt}")
            return parsed

        print(
            f"    ⚠  rows={len(services)} (expected {expected_rows}), "
            f"all_empty={not empty_ok}"
        )
        last_parsed = parsed

    print(f"  ⚠  All {MAX_RETRIES} attempts exhausted — using best available result")
    return last_parsed


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 9 — MAIN PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

def run_pipeline(pdf_path: str, output_dir: str = "EOB_OUTPUT/Medcost_new", company_name = "Medcost") -> list | None:
    """
    End-to-end pipeline for a single Medcost EOB PDF.

    Steps:
      1. Derive IDs and paths
      2. Crop claim tables (cross-page aware)
      3. Check denial status
      4. For each table: enhance → run VLM → normalise → validate
      5. Bundle and save JSON
    """

    # ── 1. Paths ───────────────────────────────────────────────────────────────
    pdf_name         = Path(pdf_path).stem                     # e.g. Pmt_EOP_364657626
    eob_id           = pdf_name.split("_")[-1]                 # e.g. 364657626
    base_dir         = os.path.join(output_dir, eob_id)
    cropped_dir      = os.path.join(base_dir, "cropped_images")
    json_output_path = os.path.join(base_dir, f"{eob_id}_output.json")
    pdf_full_name = os.path.basename(pdf_path)


    from output_utils import SUCCESS_DIR, FAILED_DIR
    already_success = os.path.exists(os.path.join(SUCCESS_DIR, company_name, pdf_name, f"{pdf_name}_output.json"))
    already_failed  = os.path.exists(os.path.join(FAILED_DIR, company_name, pdf_name, f"{pdf_name}_output.json"))
    if already_success or already_failed:
        print(f"⏭️  Skipping {pdf_name} — output already exists")
        return None

    os.makedirs(base_dir,    exist_ok=True)
    os.makedirs(cropped_dir, exist_ok=True)

    print(f"\n{'═'*65}")
    print(f"  Processing EOB: {eob_id}  ({pdf_path})")
    print(f"{'═'*65}")

    # ── 2. Crop tables ─────────────────────────────────────────────────────────
    print("\n[1/4] Cropping claim tables …")
    cropped_items = crop_medcost_claim_tables(pdf_path, output_dir=cropped_dir)

    if not cropped_items:
        print("  ⚠  No claim tables found — aborting.")
        return None

    # ── 3. Denial check ────────────────────────────────────────────────────────
    print("\n[2/4] Checking denial status …")
    claim_status = check_claim_denied(pdf_path)
    print(f"  Claim status: {claim_status}")

    # ── 4. Inference ───────────────────────────────────────────────────────────
    print(f"\n[3/4] Running VLM on {len(cropped_items)} table(s) …")
    prompt  = build_prompt(eob_id)
    results = []

    for idx, item in enumerate(cropped_items, start=1):
        img_path      = item["image_path"]
        expected_rows = item["expected_rows"]

        print(f"\n  ── Table {idx}/{len(cropped_items)}  ({img_path})  expected_rows={expected_rows}")

        # Enhance
        image_pil = enhance_table_image(img_path)

        # Infer
        parsed = run_model_with_retry(image_pil, prompt, expected_rows)

        if parsed is None:
            print(f"  ❌ Table {idx} skipped — model returned nothing usable")
            continue

        # Normalise amounts
        parsed = normalize_amounts(parsed)
        parsed["_expected_rows"] = expected_rows

        print("\n  Extracted JSON:")
        print(json.dumps(parsed, indent=4))

        results.append(parsed)

    # ── 5. Validation & final bundle ───────────────────────────────────────────
    print(f"\n[4/4] Validating {len(results)} extracted claim(s) …")

    confidence_results = []  

    for patient in results:
        is_valid, errors, total_fields = validate_claim(
            patient=patient,
            expected_row_count=patient.get("_expected_rows", 0),
        )
        patient["validation"] = {
            "status":  is_valid,
            "errors": errors,
        }
        patient["_total_fields"] = total_fields   # NEW — keep, don't skip
        confidence_results.append(patient) 

    confidence_score = calculate_eob_confidence(confidence_results)


    # Remove internal tracking key before saving
    for p in results:
        p.pop("_expected_rows", None)
        p.pop("_total_fields", None)
        p.pop("_model_confidence", None)

    final_output = [
        {
            "eob_id":       eob_id,
            "file_name":pdf_full_name,
            "claim_status": claim_status,
            "payor": "MEDCOST BENEFIT SERVICES",
            "confidence_score": confidence_score, 
            "patients":     results,
        }
    ]

    success_path, failed_path = save_split_output(
                                    final_output,
                                    company_name=company_name,
                                    pdf_name=pdf_name,
                                    pdf_path=pdf_path,
                                    cropped_dir=cropped_dir,
                                )
                            
    print(f"\n📁 Cropped images : {cropped_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final_output


W0901 19:09:50.989000 3510866 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 19:09:51.005000 3510866 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [3]:
run_pipeline(r"/home/cipl/users/OCR_Project/Medcost/pdfs/Pmt_EOP_364657626.pdf")


═════════════════════════════════════════════════════════════════
  Processing EOB: 364657626  (/home/cipl/users/OCR_Project/Medcost/pdfs/Pmt_EOP_364657626.pdf)
═════════════════════════════════════════════════════════════════

[1/4] Cropping claim tables …
  Page 1 → starts: [308.35402, 463.32979, 640.9825900000001]  ends: [453.64077, 631.29357]
  Page 2 → starts: [55.02376000000004]  ends: [177.63326000000006]
  Claim 1: p1 y=308 → p1 y=454
    ✔  Saved: EOB_OUTPUT/Medcost_new/364657626/cropped_images/page1_claim1.png  (expected_rows=3)
  Claim 2: p1 y=463 → p1 y=631
    ✔  Saved: EOB_OUTPUT/Medcost_new/364657626/cropped_images/page1_claim2.png  (expected_rows=5)
  Claim 3: p1 y=641 → p2 y=178
    ↳ Stitching 2 slices
    ✔  Saved: EOB_OUTPUT/Medcost_new/364657626/cropped_images/page1_claim3.png  (expected_rows=3)
  ⚠  No end found for claim 4 (page 2)

[2/4] Checking denial status …


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


  Claim status: not denied

[3/4] Running VLM on 3 table(s) …

  ── Table 1/3  (EOB_OUTPUT/Medcost_new/364657626/cropped_images/page1_claim1.png)  expected_rows=3
  🔄 Model attempt 1/2 …


[transformers] Both `max_new_tokens` (=15000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=15000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    ✅ Accepted on attempt 1

  Extracted JSON:
{
    "patient_name": "ONI, ANIKE I.",
    "provider": "DUC TANG DDS PLLC",
    "services": [
        {
            "service_date": "06/12/23",
            "service_code": "D1120",
            "total_charge": "82.58",
            "in_network_adjustment": "0.00",
            "not_covered": "2.37",
            "deductible": "0.00",
            "co_pay": "0.00",
            "coinsurance": "0.00",
            "plan_payment_amount": "$80.21"
        },
        {
            "service_date": "06/12/23",
            "service_code": "D0120",
            "total_charge": "65.63",
            "in_network_adjustment": "0.00",
            "not_covered": "1.51",
            "deductible": "0.00",
            "co_pay": "0.00",
            "coinsurance": "0.00",
            "plan_payment_amount": "$64.12"
        },
        {
            "service_date": "06/12/23",
            "service_code": "D1206",
            "total_charge": "53.08",
            "in_net

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=15000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    ✅ Accepted on attempt 1

  Extracted JSON:
{
    "patient_name": "ONI, ADELEKE",
    "provider": "DUC TANG DDS PLLC",
    "services": [
        {
            "service_date": "06/12/23",
            "service_code": "D0230",
            "total_charge": "32.83",
            "in_network_adjustment": "0.00",
            "not_covered": "0.37",
            "deductible": "0.00",
            "co_pay": "0.00",
            "coinsurance": "0.00",
            "plan_payment_amount": "$32.46"
        },
        {
            "service_date": "06/12/23",
            "service_code": "D0220",
            "total_charge": "39.13",
            "in_network_adjustment": "0.00",
            "not_covered": "3.06",
            "deductible": "0.00",
            "co_pay": "0.00",
            "coinsurance": "0.00",
            "plan_payment_amount": "$36.07"
        },
        {
            "service_date": "06/12/23",
            "service_code": "D0274",
            "total_charge": "84.57",
            "in_netw

[{'eob_id': '364657626',
  'file_name': 'Pmt_EOP_364657626.pdf',
  'claim_status': 'not denied',
  'payor': 'MEDCOST BENEFIT SERVICES',
  'confidence_score': 99.0,
  'patients': [{'patient_name': 'ONI, ANIKE I.',
    'provider': 'DUC TANG DDS PLLC',
    'services': [{'service_date': '06/12/23',
      'service_code': 'D1120',
      'total_charge': '82.58',
      'in_network_adjustment': '0.00',
      'not_covered': '2.37',
      'deductible': '0.00',
      'co_pay': '0.00',
      'coinsurance': '0.00',
      'plan_payment_amount': '$80.21'},
     {'service_date': '06/12/23',
      'service_code': 'D0120',
      'total_charge': '65.63',
      'in_network_adjustment': '0.00',
      'not_covered': '1.51',
      'deductible': '0.00',
      'co_pay': '0.00',
      'coinsurance': '0.00',
      'plan_payment_amount': '$64.12'},
     {'service_date': '06/12/23',
      'service_code': 'D1206',
      'total_charge': '53.08',
      'in_network_adjustment': '0.00',
      'not_covered': '0.00',
    

In [ ]:
run_pipeline(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/Medcost/pdf/Pmt_EOP_386554933.pdf")


═════════════════════════════════════════════════════════════════
  Processing EOB: 386554933  (/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/Medcost/pdf/Pmt_EOP_386554933.pdf)
═════════════════════════════════════════════════════════════════

[1/4] Cropping claim tables …
  Page 1 → starts: [336.42366, 513.38661]  ends: [504.38759, 647.33465]
  Claim 1: p1 y=336 → p1 y=504
    ✔  Saved: EOB_OUTPUT/Medcost_new/386554933/cropped_images/page1_claim1.png  (expected_rows=5)
  Claim 2: p1 y=513 → p1 y=647
    ✔  Saved: EOB_OUTPUT/Medcost_new/386554933/cropped_images/page1_claim2.png  (expected_rows=2)

[2/4] Checking denial status …


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=15000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Claim status: not denied

[3/4] Running VLM on 2 table(s) …

  ── Table 1/2  (EOB_OUTPUT/Medcost_new/386554933/cropped_images/page1_claim1.png)  expected_rows=5
  🔄 Model attempt 1/2 …


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=15000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    ✅ Accepted on attempt 1

  Extracted JSON:
{
    "patient_name": "DAVID A WILLIAMS",
    "provider": "DUC TANG D.D.S",
    "services": [
        {
            "service_date": "01/10/23",
            "service_code": "D0150",
            "total_charge": "112.34",
            "in_network_adjustment": "0.00",
            "not_covered": "0.00",
            "deductible": "0.00",
            "co_pay": "0.00",
            "coinsurance": "0.00",
            "plan_payment_amount": "$112.34"
        },
        {
            "service_date": "01/10/23",
            "service_code": "D0274",
            "total_charge": "84.57",
            "in_network_adjustment": "0.00",
            "not_covered": "0.00",
            "deductible": "0.00",
            "co_pay": "0.00",
            "coinsurance": "0.00",
            "plan_payment_amount": "$84.57"
        },
        {
            "service_date": "01/10/23",
            "service_code": "D0220",
            "total_charge": "39.13",
            "in_n

[{'eob_id': '386554933',
  'claim_status': 'not denied',
  'payor': 'MEDCOST BENEFIT SERVICES',
  'confidence_score': 99.0,
  'patients': [{'patient_name': 'DAVID A WILLIAMS',
    'provider': 'DUC TANG D.D.S',
    'services': [{'service_date': '01/10/23',
      'service_code': 'D0150',
      'total_charge': '112.34',
      'in_network_adjustment': '0.00',
      'not_covered': '0.00',
      'deductible': '0.00',
      'co_pay': '0.00',
      'coinsurance': '0.00',
      'plan_payment_amount': '$112.34'},
     {'service_date': '01/10/23',
      'service_code': 'D0274',
      'total_charge': '84.57',
      'in_network_adjustment': '0.00',
      'not_covered': '0.00',
      'deductible': '0.00',
      'co_pay': '0.00',
      'coinsurance': '0.00',
      'plan_payment_amount': '$84.57'},
     {'service_date': '01/10/23',
      'service_code': 'D0220',
      'total_charge': '39.13',
      'in_network_adjustment': '0.00',
      'not_covered': '0.00',
      'deductible': '0.00',
      'co_pay'

: 